In [ ]:
import pyActigraphy
import plotly.graph_objects as go
import os
import pandas as pd
import numpy as np
import os

In [ ]:
fpath = 'C:\\Users\\gg00642\\OneDrive - University of Surrey\\Desktop\\Actigraphy Sara'

In [ ]:
raw = pyActigraphy.io.read_raw_mtn(fpath + '\\Joined_21-09-22_25-08-24.mtn')

In [ ]:
raw.name

In [ ]:
raw.start_time

In [ ]:
raw.duration()

VISUALIZATION

In [ ]:
raw.light

In [ ]:
raw.light.get_channel_list()

In [ ]:
layout = go.Layout(
    title="Light exposure data",
    xaxis=dict(title="Date time"),
    yaxis=dict(title="Lux(log scale)"),
    showlegend=False
)
fig1 = go.Figure(
    data=[go.Scatter(
        x=raw.light.get_channel('whitelight').index.astype(str),
        y=raw.light.get_channel('whitelight'),
        name='White light')
    ],
    layout=layout
)
fig1.show()

In [ ]:
raw.light.create_light_mask()

In [ ]:
# manually adding mask to more than one period (FOR FIRST JOINED FILED (SEPT 2022 TO AUGUST 2024))
periods = [{'start': '2022-12-28 12:00:00', 'stop': '2022-12-29 12:00:00'},{'start': '2023-03-04 22:40:00', 'stop': '2023-04-18 10:00:00'},{'start': '2023-05-14 22:30:00', 'stop': '2023-05-15 07:15:00'},
           {'start': '2023-05-16 09:00:00', 'stop': '2023-06-29 13:00:00'},{'start': '2023-07-31 19:15:00', 'stop': '2023-08-16 12:00:00'},{'start': '2023-08-18 10:00:00', 'stop': '2023-08-22 13:00:00'},
           {'start': '2024-06-27 10:52:00', 'stop': '2024-06-28 10:40:00'}, {'start': '2024-07-30 19:15:00', 'stop': '2024-08-04 19:15:00'}]
for period in periods:
    raw.light.add_light_mask_period(start=period['start'], stop=period['stop'], channel = "whitelight")

In [ ]:
# manually adding mask to more than one period
periods = [{'start': '2023-03-04 22:00:00', 'stop': '2023-04-18 09:15:00'},{'start': '2023-05-16 06:30:00', 'stop': '2023-06-29 16:00:00'},{'start': '2023-07-28 14:00:00', 'stop': '2023-08-22 16:00:00'}]
for period in periods:
    raw.light.add_light_mask_period(start=period['start'], stop=period['stop'], channel = "whitelight")

In [ ]:
# manually adding mask to more than one period (THIRD JOINED FILE)
periods = [{'start': '2024-12-18 22:00:00', 'stop': '2024-12-19 07:10:00'},{'start': '2025-01-02 13:00:00', 'stop': '2025-01-03 09:00:00'},{'start': '2025-01-31 17:00:00', 'stop': '2025-02-03 13:15:00'}]
for period in periods:
    raw.light.add_light_mask_period(start=period['start'], stop=period['stop'], channel = "whitelight")

In [ ]:
raw.light.apply_mask = True

In [ ]:
layout = go.Layout(
    xaxis=dict(title="Date time"),
    yaxis=dict(title="Activity counts/period"),
    yaxis2=dict(title='Mask',overlaying='y',side='right'),
    showlegend=True
)

In [ ]:
fig2 = go.Figure([
    go.Scatter(
        x=raw.light.get_channel('whitelight').index.astype(str),
        y=raw.light.get_channel('whitelight'),
        name='whitelight'),
    go.Scatter(
        x=raw.light.mask.index.astype(str),
        y=raw.light.mask.loc[:,'whitelight'],
        yaxis='y2', opacity=0.5,
        name='Mask')
], layout=layout)

In [ ]:
fig2.show()

In [ ]:
#Light data are (log10+1)-transformed in pyActigraphy. Therefore, when threshold values are meant to applied to the light levels, the corresponding value on the (log10+1) scale should be applied.

#NB: an offset of 1 is added to the light data before log10 transformation in order to avoid a divergence of the log10 function when the light data values are zero: $:nbsphinx-math:log_{10}(0) = -:nbsphinx-math:`infty `$.
#So, on a (log10+1) scale:
#10 lux threshold correspond to a value of log(10+1)~1
#100 lux threshold correspond to a value of log(100+1)~2
#1000 lux threshold correspond to a value of log(1000+1)~3

#To simply get the exact value, use the np.log10 function of the numpy package we imported earlier:

In [ ]:
np.log10(10+1), np.log10(100+1), np.log10(1000+1)

In [ ]:
# converting 250 melanopic lux to photopic lux (not that simple) - photopic lux = melanopic EDI/melanopic ratio
# standard office lighting (3500 K LED) with 300 lux at the task plane and 150 lux at the eye has a melanopic ratio (melanopic DER) of about 0.51 (which I used here to convert)
# REF: https://pmc.ncbi.nlm.nih.gov/articles/PMC8215265/
# also here: https://www.fagerhult.com/knowledge/light-and-people/melanopic-ratio/a-new-way-to-report-the-biological-impact-of-lighting/#:~:text=Melanopic%20Ratio%20(Melanopic%20Daylight%20Efficacy,the%20reference%20for%20Melanopic%20Ratio.
# https://journals.sagepub.com/doi/epub/10.1177/14771535221120350
# https://issuu.com/designinglighting/docs/dec_2022/s/18078622

# conversion: photopic lux = 250 melanopic EDI / 0.51 (melanopic DER) = 490 lux
# converted to log scale (to use it in the TAT treshold): np.log10(490+1) = 2.69

# treshold for outdoor light exposure = 2500 lux
# converted to log scale: 3.398

In [ ]:
np.log10(490+1)

In [ ]:
np.log10(2500+1)

_exposure level_

In [ ]:
help(raw.light.light_exposure_level)

In [ ]:
raw.light.light_exposure_level(agg='median')

In [ ]:
raw.light.light_exposure_level(
    threshold=1, # on a log10 scale, 2 means 10^2 and in lux it is 10^2 lux
    start_time='8:00',
    stop_time='18:00'
)

_Summary statistics (mean, median, s.d., max., min., sum)_

In [ ]:
help(raw.light.summary_statistics_per_time_bin)

In [ ]:
a = raw.light.summary_statistics_per_time_bin()

In [ ]:
raw.light.summary_statistics_per_time_bin(bins='12h')

In [ ]:
raw.light.summary_statistics_per_time_bin(
    bins=[
        ['2023-09-19 12:00:00','2023-09-19 19:59:00'],
        ['2023-09-23 12:00:00','2023-09-23 23:59:00']
    ],
    agg_func=['mean','std']
)

_Time above threshold (TAT)_

In [ ]:
help(raw.light.TAT)

In [ ]:
raw.light.TAT() # the results are identical to the total number of epochs in the recording.

In [ ]:
raw.light.TAT(threshold=2) # with a threshold of 2 this means that the light intensity must be greater than 10^2 lux to be considered as light exposure.

In [ ]:
# The parameter oformat defines the output format. Available formats: * ‘minute’:
raw.light.TAT(threshold=2, oformat='minute') 

In [ ]:
raw.light.TAT(threshold=2, oformat='timedelta') # the output is a timedelta object

In [ ]:
# Time spent above threshold at specific time periods:
raw.light.TAT(
    threshold=2, start_time='08:00:00', stop_time='20:00:00', oformat='timedelta'
)

_Time above threshold per period (TATp)_

In [ ]:
help(raw.light.TATp)

In [ ]:
raw.light.TATp(threshold=2, oformat='timedelta')

_Mean light timing (MLit)_

In [ ]:
help(raw.light.MLiT)

In [ ]:
# The light exposure data are converted to log10+1.
raw.light.MLiT(threshold=np.log10(500+1))

In [ ]:
# To convert this number of minutes into a time of day, simply divide it by the number of minutes per hour:
divmod(812.33715,60) # The mean light timing for xxx light exposure above 500 lux is around h, min for this recording.

In [ ]:
raw.light.MLiTp(threshold=np.log10(500+1))

In [ ]:
mlitp_value = raw.light.MLiTp(threshold=np.log10(500 + 1))

# Convert into hours and minutes
h, min = divmod(mlitp_value, 60)

_L5 and M10 (values and timing)_

In [ ]:
help(raw.light.LMX)

In [ ]:
# Remove the NaN values
#cleaned_data = raw.light.data.dropna()

# Update the raw.light object with the cleaned data
#raw.light._data = cleaned_data

In [ ]:
raw.light.LMX(length='10h',lowest=False)

In [ ]:
raw.light.LMX(length='5h',lowest=True)

In [ ]:
dlp = raw.light.average_daily_profile(
    rsfreq='60min', cyclic=False,# rsfreq = resampling frequency (in minutes) to compute the average daily profile 
    channel='whitelight', # channel to use for the computation
)

In [ ]:
dlp_fig = go.Figure(go.Scatter(x=dlp.index.astype(str),y=dlp))

In [ ]:
dlp_fig.show()


_Access to raw and thresholded data_

In [ ]:
c = raw.light.data

In [ ]:
# divide b in chunck according to the year that you can extract from the date
c['year'] = c.index.year

In [ ]:
# devide the dataframe b in different dataframes according to the year column and name separatly
c_2022 = c[c['year'] == 2022]
c_2023 = c[c['year'] == 2023]
c_2024 = c[c['year'] == 2024]

In [ ]:
# convert the data from log to lux
c_2022['whitelight'] = 10**c_2022['whitelight'] - 1
c_2023['whitelight'] = 10**c_2023['whitelight'] - 1
c_2024['whitelight'] = 10**c_2024['whitelight'] - 1

In [ ]:
# make the index of b as first column 
c_2022.reset_index(inplace=True)
c_2023.reset_index(inplace=True)
c_2024.reset_index(inplace=True)

In [ ]:
# rename index column to date
c_2022.rename(columns={'index':'date'}, inplace=True)
c_2023.rename(columns={'index':'date'}, inplace=True)
c_2024.rename(columns={'index':'date'}, inplace=True)

In [ ]:
# average the withelight data every 30 minutes
c_2022['date'] = pd.to_datetime(c_2022['date'])
c_2023['date'] = pd.to_datetime(c_2023['date'])
c_2024['date'] = pd.to_datetime(c_2024['date'])

c_2022.set_index('date', inplace=True)
c_2023.set_index('date', inplace=True)
c_2024.set_index('date', inplace=True)

c_2022 = c_2022.resample('30T').mean()
c_2023 = c_2023.resample('30T').mean()
c_2024 = c_2024.resample('30T').mean()

In [ ]:
# cancel the column whitelight, time_bin, and year
c_2022.drop(columns=['year'], inplace=True)
c_2023.drop(columns=['year'], inplace=True)
c_2024.drop(columns=['year'], inplace=True)

In [ ]:
# download the data
c_2022.to_csv('C:\\Users\\gg00642\\OneDrive - University of Surrey\\Desktop\\Actigraphy Sara\\light_2022.csv')
c_2023.to_csv('C:\\Users\\gg00642\\OneDrive - University of Surrey\\Desktop\\Actigraphy Sara\\light_2023.csv')
c_2024.to_csv('C:\\Users\\gg00642\\OneDrive - University of Surrey\\Desktop\\Actigraphy Sara\\light_2024.csv')